# 2.1. Gom tệp nguồn đa tệp và dựng cây cấu trúc phân cấp

## 1. Xác định tệp gốc biên dịch và mở rộng `\input` / `\include`

- **Phân giải tệp gốc khi có nhiều ứng viên (`gather.py`)**: Sau khi loại bỏ chú thích `%` (có bảo vệ ký tự thoát `\%`, `\verb|...|` và các môi trường nguyên văn như `verbatim`, `lstlisting`, `minted`), trình quét tìm các tệp chứa `\documentclass` hoặc `\documentstyle`. Khi một gói nguồn có nhiều tệp ứng viên (ví dụ đi kèm `supplement.tex` hoặc `rebuttal.tex`), hệ thống xét theo thứ tự: cấu hình [`configs/parser_roots.json`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/configs/parser_roots.json), ứng viên duy nhất, và sau cùng là đối sánh độ trùng khớp từ khóa trong `\title{...}` với tiêu đề bài báo ở M1 (giúp xác định đúng tệp gốc cho 1.706 phiên bản đa tệp).
- **Quy tắc dựng cây phân cấp (`hierarchy.py`)**: Cây bắt đầu từ nút gốc `document`, phân nhánh xuống `chapter`/`section`/`subsection`..., `paragraph`, `list` và `item`, cùng ba loại nút lá nội dung là `sentence`, `block_formula` và `figure` (gộp chung `figure`, `figure*`, `table`, `table*`). Khi gặp đề mục tài liệu tham khảo (`References`, `Bibliography`), biến trạng thái `skip_level` loại bỏ phần danh mục tham khảo khỏi cây nội dung nhưng tự động khôi phục khi gặp đề mục ngang cấp hoặc `\appendix` ở phía sau, nhờ đó giữ đầy đủ `Acknowledgements` và `Appendix`.

## 2. Minh họa trực tiếp quá trình gom tệp và dựng cây phân cấp từ mã nguồn

In [ ]:
import html
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, Rectangle
from matplotlib.ticker import FuncFormatter
from IPython.display import display, Markdown, HTML

data = json.loads((Path.cwd().parent / "report_summary.json").read_text(encoding="utf-8"))
scope, m1, m2, bench = data["scope"], data["m1"], data["m2"], data["m2_benchmark"]

# Bảng màu học thuật tối giản: 1 tông xanh đậm chủ đạo kết hợp thang xám điềm đạm
BOOK_PALETTE = {
    "navy": "#1B365D",
    "steel": "#3B6E8C",
    "slate": "#64748B",
    "muted": "#94A3B8",
    "light": "#CBD5E1",
    "accent": "#2C5282",
    "grid": "#CBD5E1",
}
MEMBER_STYLES = {
    "DAT": ("#1B365D", "-"),
    "TUAN": ("#3B6E8C", "--"),
    "TAN": ("#4A5568", "-."),
    "TRI": ("#64748B", ":"),
    "TRIET": ("#94A3B8", "-"),
}

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Liberation Sans", "DejaVu Sans", "Arial"],
    "figure.figsize": (9.6, 4.0),
    "figure.dpi": 170,
    "savefig.dpi": 170,
    "savefig.bbox": "tight",
    "figure.facecolor": "#FFFFFF",
    "axes.facecolor": "#FFFFFF",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": "#94A3B8",
    "axes.linewidth": 0.85,
    "axes.titlesize": 10.5,
    "axes.titleweight": "bold",
    "axes.titlecolor": "#1B365D",
    "axes.labelsize": 9.2,
    "axes.labelcolor": "#1E293B",
    "xtick.labelsize": 8.6,
    "ytick.labelsize": 8.6,
    "xtick.color": "#334155",
    "ytick.color": "#334155",
    "grid.color": "#CBD5E1",
    "grid.linestyle": ":",
    "grid.alpha": 0.50,
})


def fmt_int(x) -> str:
    if x is None or pd.isna(x):
        return "-"
    return f"{int(round(float(x))):,}".replace(",", ".")


def fmt_num(x, digits: int = 2) -> str:
    if x is None or pd.isna(x):
        return "-"
    return f"{float(x):,.{digits}f}".replace(",", "X").replace(".", ",").replace("X", ".")


def fmt_pct(x, digits: int = 2, scale100: bool = False) -> str:
    if x is None or pd.isna(x):
        return "-"
    val = float(x) * (100.0 if scale100 else 1.0)
    return f"{val:.{digits}f}%".replace(".", ",")


def style_table(
    df: pd.DataFrame,
    caption: str | None = None,
    precision: int = 2,
    int_cols: list[str] | None = None,
    pct_cols: list[str] | None = None,
    pct_scale100: bool = False,
    col_formats: dict[str, object] | None = None,
    **_unused,
):
    # Bảng học thuật kiểu booktabs tối giản: không thanh màu nền, không nút sắp xếp, an toàn HTML 100%.
    out = df.copy()
    int_set = set(int_cols or [])
    pct_set = set(pct_cols or [])
    custom_fmt = dict(col_formats or {})

    num_cols: set[str] = set()
    for c in out.columns:
        if c in int_set or c in pct_set or pd.api.types.is_numeric_dtype(out[c]):
            num_cols.add(c)

    parts = ['<div class="ah-table-scroll"><table class="dataframe ah-clean-table">']
    if caption:
        parts.append(f"<caption>{html.escape(str(caption))}</caption>")

    parts.append("<thead><tr>")
    for col in out.columns:
        cls = ' class="num-col"' if col in num_cols else ""
        parts.append(f"<th{cls}>{html.escape(str(col))}</th>")
    parts.append("</tr></thead><tbody>")

    for _, row in out.iterrows():
        parts.append("<tr>")
        for col in out.columns:
            val = row[col]
            if col in custom_fmt:
                txt = custom_fmt[col](val)
            elif isinstance(val, (list, tuple)):
                txt = ", ".join(str(x) for x in val) if val else "-"
            elif val is None or (not isinstance(val, (str, bytes)) and pd.isna(val)):
                txt = "-"
            elif col in int_set or pd.api.types.is_integer_dtype(out[col]):
                txt = fmt_int(val)
            elif col in pct_set:
                txt = fmt_pct(val, precision, pct_scale100)
            elif pd.api.types.is_float_dtype(out[col]):
                txt = fmt_num(val, precision)
            else:
                txt = str(val)
            cls = ' class="num-col"' if col in num_cols else ""
            esc = html.escape(str(txt))
            if not cls and str(txt).startswith(("http://", "https://", "10.")):
                esc = f'<span class="ah-breakable">{esc}</span>'
            parts.append(f"<td{cls}>{esc}</td>")
        parts.append("</tr>")

    parts.append("</tbody></table></div>")
    return HTML("".join(parts))


def draw_pipeline_diagram(title: str, stages: list[tuple[str, str]]) -> None:
    # Sơ đồ khối quy trình tối giản chuẩn báo cáo khoa học.
    fig, ax = plt.subplots(figsize=(10.0, 2.05))
    ax.set_xlim(0, 10.0)
    ax.set_ylim(0, 2.05)
    ax.axis("off")
    n = len(stages)
    bw = 2.22
    gap = (9.70 - n * bw) / max(n - 1, 1)
    for idx, (st_title, st_desc) in enumerate(stages):
        x0 = 0.15 + idx * (bw + gap)
        ax.add_patch(Rectangle((x0, 0.18), bw, 1.56, facecolor="#FFFFFF", edgecolor="#CBD5E1", linewidth=1.0))
        ax.add_patch(Rectangle((x0, 1.32), bw, 0.42, facecolor="#EDF2F7", edgecolor="#CBD5E1", linewidth=0.8))
        ax.add_patch(Rectangle((x0, 1.32), 0.06, 0.42, facecolor="#1B365D", edgecolor="none"))
        ax.text(x0 + bw / 2 + 0.02, 1.53, st_title, ha="center", va="center", color="#1B365D", fontsize=8.8, fontweight="bold")
        ax.text(x0 + bw / 2, 0.74, st_desc, ha="center", va="center", color="#334155", fontsize=8.0, linespacing=1.35)
        if idx < n - 1:
            ax.add_patch(FancyArrowPatch(
                (x0 + bw + 0.04, 0.96),
                (x0 + bw + gap - 0.04, 0.96),
                arrowstyle="-|>,head_length=5,head_width=3.2",
                color="#64748B",
                lw=1.1,
            ))
    ax.set_title(title, pad=4, fontsize=10.0, fontweight="bold", color="#1B365D")
    plt.tight_layout()
    plt.show()

In [ ]:
import tempfile
from collections import defaultdict
from parser.gather import gather
from parser.hierarchy import parse_hierarchy

main_tex = r"""\documentclass{article}
\title{Demo Multi-file Paper}
\begin{document}
% \input{ignored_comment}
\input{sections/intro}
\section*{Acknowledgements}
We thank the team.
\appendix
\section{Appendix Proof}
Extra proof detail.
\section*{References}
\begin{thebibliography}{9}
\bibitem{k1} A. Author. Title. 2020.
\end{thebibliography}
\end{document}"""

intro_tex = r"""\section{Introduction}
First sentence with Fig. 1 and value 3.14.
\begin{itemize}\item First point.\item Second point.\end{itemize}
\begin{align}x + y = z\end{align}
\begin{table}[htpb]\centering\begin{tabular}{cc}\toprule A & B \\ \midrule 1 & 2 \\ \bottomrule\end{tabular}\end{table}
"""

with tempfile.TemporaryDirectory() as tmp:
    root = Path(tmp)
    (root / "sections").mkdir()
    (root / "main.tex").write_text(main_tex, encoding="utf-8")
    (root / "sections/intro.tex").write_text(intro_tex, encoding="utf-8")
    (root / "redundant.tex").write_text("Ignored standalone file.", encoding="utf-8")
    g = gather(root)
    tree = parse_hierarchy(g.text)

df_ast = pd.DataFrame([
    {"node_id": n.node_id, "parent_id": -1 if n.parent_id is None else n.parent_id, "kind": n.kind, "content": n.content[:85]}
    for n in tree.nodes
])
display(style_table(df_ast, caption="Bảng 6.1. Danh sách nút cây cấu trúc trích xuất từ bài báo đa tệp minh họa (bỏ qua tệp rời redundant.tex và mục References).", int_cols=["node_id", "parent_id"]))

children = defaultdict(list)
by_id = {n.node_id: n for n in tree.nodes}
for n in tree.nodes:
    if n.parent_id is not None:
        children[n.parent_id].append(n.node_id)

pos, leaf_cursor = {}, [0]
def _layout_ast(nid: int, depth: int = 0) -> float:
    ch = children.get(nid, [])
    if not ch:
        y = -float(leaf_cursor[0])
        leaf_cursor[0] += 1
    else:
        ys = [_layout_ast(c, depth + 1) for c in ch]
        y = sum(ys) / len(ys)
    pos[nid] = (depth, y)
    return y
_layout_ast(0)

fig, ax = plt.subplots(figsize=(10.2, 5.2))
x_step = 2.35
for nid, (d, y) in pos.items():
    x = d * x_step
    for cid in children.get(nid, []):
        cd, cy = pos[cid]
        ax.plot([x + 1.86, cd * x_step - 0.02], [y, cy], color="#94A3B8", lw=1.0, zorder=1)
    n = by_id[nid]
    snippet = n.content.replace("\n", " ").strip()
    snippet = (snippet[:24] + "...") if len(snippet) > 25 else snippet
    label = f"[{n.node_id}] {n.kind}\n{snippet}"
    is_leaf = n.kind in ("sentence", "block_formula", "figure")
    bg = "#EDF2F7" if is_leaf else "#FFFFFF"
    ec = "#3B6E8C" if is_leaf else "#1B365D"
    ax.add_patch(Rectangle((x, y - 0.35), 1.86, 0.70, facecolor=bg, edgecolor=ec, lw=0.95, zorder=2))
    ax.text(x + 0.93, y, label, ha="center", va="center", color="#1E293B", fontsize=7.4, zorder=3)

max_d = max(d for d, _ in pos.values())
ax.set_xlim(-0.15, (max_d + 1) * x_step)
ax.set_ylim(-leaf_cursor[0] + 0.2, 0.8)
ax.axis("off")
ax.set_title("Hình 6.1. Sơ đồ cây phân cấp sinh tự động từ đối tượng tree.nodes", pad=8)
plt.tight_layout()
plt.show()

## 3. Thống kê chọn tệp gốc, so sánh thử nghiệm Pilot và phân bố 15.006.000 phần tử

In [ ]:
gs = m2["gathering_stats"]
rs_df = pd.DataFrame([
    {"Cơ chế chọn tệp gốc biên dịch": str(k), "Số phiên bản TeX": int(v), "Tỷ lệ / 38.554 bản TeX (%)": 100.0 * v / m2["versions_with_tex"]}
    for k, v in gs["root_selection"].items()
])
display(style_table(
    rs_df,
    caption=f"Bảng 6.2. Phân bố cơ chế xác định tệp gốc trên 38.554 phiên bản TeX (nhiều nhất {gs['max_comp_files']} tệp .tex trong một phiên bản).",
    int_cols=["Số phiên bản TeX"],
    pct_cols=["Tỷ lệ / 38.554 bản TeX (%)"],
))

pv = m2["pilot_v1_vs_v2"]
all_st_keys = ["success", "partial", "empty_no_tex", "error", "error_missing_source"]
comp = pd.DataFrame([
    {
        "Trạng thái": k,
        "Pilot v1 (trước cải tiến)": int(pv["v1"]["status_counts"].get(k, 0)),
        "Pilot v2 (sau cải tiến)": int(pv["v2"]["status_counts"].get(k, 0)),
        "Chênh lệch": int(pv["v2"]["status_counts"].get(k, 0)) - int(pv["v1"]["status_counts"].get(k, 0)),
    }
    for k in all_st_keys
])
display(style_table(
    comp,
    caption="Bảng 6.3. Kết quả đối chứng trên tập Pilot 250 bài (536 phiên bản) trước và sau khi bổ sung xử lý ngoại lệ LaTeX (giảm 80,0% số phiên bản lỗi).",
    int_cols=["Pilot v1 (trước cải tiến)", "Pilot v2 (sau cải tiến)", "Chênh lệch"],
))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10.4, 3.8))
xp = np.arange(len(comp))
wp = 0.35
ax1.bar(xp - wp / 2, comp["Pilot v1 (trước cải tiến)"].to_numpy(dtype=float), wp, color=BOOK_PALETTE["slate"], label="Pilot v1 (72,6% success)")
ax1.bar(xp + wp / 2, comp["Pilot v2 (sau cải tiến)"].to_numpy(dtype=float), wp, color=BOOK_PALETTE["navy"], label="Pilot v2 (83,9% success)")
ax1.set_xticks(xp)
ax1.set_xticklabels(["Success", "Partial", "Empty", "Error", "404"])
ax1.set_ylim(0, 470)
ax1.grid(axis="y")
ax1.legend(fontsize=8.0)
ax1.set_ylabel("Số phiên bản")
ax1.set_title("Hình 6.2a. So sánh tập Pilot 250 bài (lỗi giảm từ 70 xuống 14)")
for idx_c, r_c in comp.iterrows():
    ax1.text(idx_c + wp / 2, float(r_c["Pilot v2 (sau cải tiến)"]) + 8, str(r_c["Pilot v2 (sau cải tiến)"]),
             ha="center", fontsize=7.8)

diag_s = pd.Series(m2["diagnostic_categories"]).sort_values(ascending=True)
yd = np.arange(len(diag_s))
ax2.barh(yd, diag_s.to_numpy(dtype=float), color=BOOK_PALETTE["steel"], height=0.55)
ax2.set_yticks(yd)
ax2.set_yticklabels([str(k) for k in diag_s.index], fontsize=8.2)
ax2.set_xlim(0, float(diag_s.max()) * 1.25)
ax2.grid(axis="x")
ax2.set_xlabel("Số phiên bản ghi nhận cảnh báo")
ax2.set_title("Hình 6.2b. Phân nhóm cảnh báo trên 41.498 phiên bản")
for idx_d, v_d in enumerate(diag_s.values):
    ax2.text(float(v_d) + 90, idx_d, fmt_int(v_d), va="center", fontsize=7.9)
plt.tight_layout()
plt.show()

In [ ]:
if "element_kind_counts" in m2:
    ek = pd.Series(m2["element_kind_counts"])
    df_ek = pd.DataFrame({
        "Loại phần tử cấu trúc (kind)": [str(k) for k in ek.index],
        "Số phần tử duy nhất": ek.values,
        "Tỷ trọng (%)": 100.0 * ek.values / ek.sum(),
    })
    display(style_table(
        df_ek,
        caption="Bảng 6.4. Phân bố 13 loại nút cấu trúc trên tổng số 15.006.000 phần tử của bảng elements.",
        int_cols=["Số phần tử duy nhất"],
        pct_cols=["Tỷ trọng (%)"],
    ))
    fig, ax = plt.subplots(figsize=(9.6, 3.9))
    ek_sorted = df_ek.sort_values("Số phần tử duy nhất", ascending=True)
    yk = np.arange(len(ek_sorted))
    ax.barh(yk, ek_sorted["Số phần tử duy nhất"].to_numpy(dtype=float), color=BOOK_PALETTE["navy"], height=0.58)
    ax.set_yticks(yk)
    ax.set_yticklabels([str(v) for v in ek_sorted["Loại phần tử cấu trúc (kind)"]])
    ax.set_xscale("symlog", linthresh=100)
    ax.set_xlim(0, 3e7)
    ax.grid(axis="x")
    ax.set_xlabel("Số phần tử duy nhất sau khử trùng (thang symlog)")
    ax.set_title("Hình 6.3. Phân bố 13 loại phần tử trong bảng elements (15.006.000 dòng)")
    for i_k, r_k in enumerate(ek_sorted.to_dict("records")):
        ax.text(float(r_k["Số phần tử duy nhất"]) * 1.15 + 2, i_k,
                f"{fmt_int(r_k['Số phần tử duy nhất'])} ({fmt_pct(r_k['Tỷ trọng (%)'])})",
                va="center", fontsize=7.9)
    plt.tight_layout()
    plt.show()

## Tài liệu và mã nguồn đối chiếu

- [Đặc tả chính thức Milestone 2 - Hierarchical Parsing and Standardization](https://hcmus-cs-ta.github.io/lab-book/intro-ds/statement/parsing/).
- [arXiv API User's Manual](https://info.arxiv.org/help/api/user-manual.html), [API Terms of Use](https://info.arxiv.org/help/api/tou.html) và [OAI-PMH Cutover (`oaipmh.arxiv.org`)](https://info.arxiv.org/help/oa/index.html).
- [Semantic Scholar Academic Graph API](https://api.semanticscholar.org/api-docs/graph).
- Mã nguồn triển khai: [`src/parser/gather.py`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/src/parser/gather.py).
- Mã nguồn triển khai: [`src/parser/latex.py`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/src/parser/latex.py).
- Mã nguồn triển khai: [`src/parser/hierarchy.py`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/src/parser/hierarchy.py).
- Mã nguồn triển khai: [`src/parser/main.py`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/src/parser/main.py).
- Mã nguồn triển khai: [`configs/parser_roots.json`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/configs/parser_roots.json).